In [1]:
pip install -U langchain-openai python-dotenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.8/85.8 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 500.5/500.5 kB 12.6 MB/s eta 0:00:00
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.2.9
    Uninstalling langchain-core-1.2.9:
      Successfully uninstalled langchain-core-1.2.9


In [2]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field
import pandas as pd
from dotenv import load_dotenv
from langchain_core.output_parsers import PydanticOutputParser
import os

In [3]:
load_dotenv()

True

In [4]:
api_key = os.getenv("OPENAI_API_KEY")



In [5]:
class MussoorieSpot(BaseModel):
    name: str
    thrill: int = Field(ge=1, le=10, description="How exciting/adventurous the place is")
    seclusion: int = Field(ge=1, le=10, description="How quiet or isolated the place is")
    driving_difficulty: int = Field(ge=1, le=10, description="How hard it is to reach by car (1=easy, 10=steep/narrow)")
    crowd_density: int = Field(ge=1, le=10, description="How crowded it usually is")
    caution: str = Field(description="A specific safety warning or 'Pro Tip' for this location")

In [6]:
with open("/content/locations.txt", "r") as f:
    # Read lines and strip whitespace to get a clean list
    location_list = [line.strip() for line in f if line.strip()]

In [7]:
location_list

['Lambi Dehar Mines',
 'Pari Tibba (Fairy Hill)',
 'Sisters Bazaar Haunted House',
 'Mullingar Mansion',
 'The Savoy Hotel',
 'Landour Graveyard',
 'Barlowganj Cemetery',
 "Camel's Back Cemetery",
 'Jabarkhet Nature Reserve',
 'Jharipani Falls',
 'Mausi Falls',
 'Cloud’s End',
 'Kimadi Road',
 'Dhanaulti (Eco Park)',
 'Kanatal',
 'Bhadraj Temple',
 'Nag Tibba Trek',
 'Nag Tibba (Base)',
 'Surkanda Devi Mandir',
 'Dalai Hill',
 'George Everest House',
 'Mall Road (Casual)',
 'Char Dukan',
 'Lal Tibba',
 'Kellogg Church',
 'St. Paul’s Church',
 'Sisters Bazaar House',
 "Camel's Back Road",
 'Landour Clock Tower',
 'Gun Hill (Trolley)',
 'Kempty Falls',
 'Company Garden',
 'Bhatta Falls',
 'Christ Church',
 'Mussoorie Lake',
 'Happy Valley',
 'Shedup Choephelling Temple',
 'Santura Devi Temple',
 'Prakasheshwar Mahadev Mandir',
 'Lake Mist',
 'Benog Sanctuary',
 'Mossy Falls']

In [8]:
SYSTEM_TEMPLATE = """
You are a highly detailed travel data analyst specializing in the geography of Mussoorie, Uttarakhand.
Your task is to generate a numerical profile for a specific location.

### SCORING DEFINITIONS:
- **Thrill (1-10):** 1 = Boring/Static (Mall Road); 10 = High Adrenaline/Extreme (Paragliding, Haunted Mines).
- **Seclusion (1-10):** 1 = Urban/Crowded (Library Chowk); 10 = Total Wilderness/No People (Bhadraj Trek).
- **Driving Difficulty (1-10):** 1 = Wide, Paved Roads; 10 = Steep, Narrow, Unpaved, or Hairpin Bends (Landour Upper Mall).
- **Crowd Density (1-10):** 1 = You are the only one there; 10 = Shoulder-to-shoulder traffic (Kempty Falls in June).

### CONSTRAINTS:
1. Base your scores on peak tourist season conditions.
2. The 'caution' field must be a specific, actionable piece of advice (e.g., "Beware of monkeys," "Four-wheel drive recommended").
3. Return ONLY the JSON object.

{format_instructions}
"""

In [9]:
parser = PydanticOutputParser(pydantic_object=MussoorieSpot)

In [10]:
llm = ChatOpenAI(model="nvidia/nemotron-3-nano-30b-a3b:free",
                 openai_api_key=api_key,
                 base_url="https://openrouter.ai/api/v1")

In [13]:
chat_prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_TEMPLATE),
    ("human", "Generate the profile for this location: {location}")
])

In [14]:
chain=chat_prompt|llm|parser

In [26]:
temp=location_list[23::]

In [27]:
all_spots = []

for i in temp:
  response=chain.invoke({
      "location":i,
      "format_instructions":parser.get_format_instructions()
    })
  all_spots.append(response.model_dump())
  print("{} added".format(i))

Lal Tibba added
Kellogg Church added
St. Paul’s Church added
Sisters Bazaar House added
Camel's Back Road added
Landour Clock Tower added
Gun Hill (Trolley) added
Kempty Falls added
Company Garden added
Bhatta Falls added
Christ Church added
Mussoorie Lake added
Happy Valley added
Shedup Choephelling Temple added
Santura Devi Temple added
Prakasheshwar Mahadev Mandir added
Lake Mist added
Benog Sanctuary added
Mossy Falls added


In [16]:
all_spots

[{'name': 'Lambi Dehar Mines',
  'thrill': 8,
  'seclusion': 9,
  'driving_difficulty': 7,
  'crowd_density': 2,
  'caution': 'Use a 4x4 vehicle for the rough final stretch and carry a flashlight; tunnels are unstable and may have falling debris.'},
 {'name': 'Pari Tibba',
  'thrill': 6,
  'seclusion': 7,
  'driving_difficulty': 5,
  'crowd_density': 3,
  'caution': 'Start early, wear sturdy trekking shoes, and watch for slippery trails and sudden weather changes.'},
 {'name': 'Sisters Bazaar Haunted House',
  'thrill': 8,
  'seclusion': 4,
  'driving_difficulty': 6,
  'crowd_density': 3,
  'caution': 'Enter only during daylight; watch for loose flooring and steep stairs, and avoid night visits.'},
 {'name': 'Mullingar Mansion',
  'thrill': 6,
  'seclusion': 8,
  'driving_difficulty': 5,
  'crowd_density': 3,
  'caution': 'Watch for uneven steps and occasional monkey activity; avoid night visits.'},
 {'name': 'The Savoy Hotel',
  'thrill': 3,
  'seclusion': 4,
  'driving_difficulty': 2

In [28]:
df=pd.DataFrame(all_spots)

In [29]:
df

,name,thrill,seclusion,driving_difficulty,crowd_density,caution
0,Lal Tibba,7,5,7,6,"Drive cautiously on the steep, narrow road; we..."
1,Kellogg Church,2,4,3,6,Parking is limited; use the designated pull‑ou...
2,St. Paul’s Church,2,3,2,5,"Beware of steep, narrow steps and occasional s..."
3,Sisters Bazaar House,2,1,3,9,Watch for parked trucks and slippery cobblesto...
4,Camel's Back Road,3,4,4,6,Monkeys can be aggressive; keep food secured a...
5,Landour Clock Tower,2,2,4,9,Expect heavy pedestrian traffic and monkeys; d...
6,Gun Hill (Trolley),5,3,2,9,Check the trolley schedule ahead of time and a...
7,Kempty Falls,6,2,4,9,Wear sturdy non‑slip shoes and avoid the water...
8,Company Garden,4,5,3,6,Monkeys are attracted to food; keep snacks sea...
9,Bhatta Falls,6,5,4,6,Wear sturdy shoes; rocks become slippery after...


In [32]:
df.to_csv("mussoorie_locations",index=False)